# 03 - Modelo de referencia y decisiones basadas en costos

En esta etapa entrenamos XGBoost con el pasado, ajustamos un calibrador en validación temprana y evaluamos sus probabilidades en validación posterior. Separamos la etiqueta clasificatoria de la decisión: intervenir significa bloquear y enviar a revisión.

Aplicamos la matriz de la propuesta: intervenir cuesta `Ca` en ambas clases, aprobar fraude cuesta `TransactionAmt` y aprobar una operación legítima cuesta cero. BMR interviene si `p_i A_i > Ca`. En empate aprobamos. El corte clasificatorio 0,5 se conserva.

Las funciones económicas viven en `src/fraud_cost/costs.py`. La comparación final requiere acordar escenarios de Ca y congelar el protocolo. El notebook 04 permite explorar valores ilustrativos sin afirmar costos operativos observados.

El test permanece reservado. Sus conteos agregados fueron consultados históricamente en el EDA y al comparar gaps; declararemos esa limitación. XGBoost es un primer modelo con parámetros fijos; la búsqueda temporal de hiperparámetros aún está pendiente.


## 1. Configuración y prerrequisitos

Definimos la semilla, los parámetros del predictor y la fracción de validación destinada a calibración antes de entrenar. Esto nos permite distinguir una configuración experimental de una elección motivada por los resultados. Los controles booleanos registran cuándo habilitamos etapas; su valor no sustituye el acuerdo metodológico del equipo.


In [1]:
from pathlib import Path
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    log_loss,
    precision_score,
    recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

from fraud_cost.data import load_ieee_cis
from fraud_cost.split import validate_temporal_split
from fraud_cost.costs import bmr_actions, evaluate_policies, fit_cost_tree, require_approved_costs


def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "data").is_dir() and (candidate / "src").is_dir():
            return candidate
    raise FileNotFoundError("Abra Jupyter desde el repositorio o una de sus subcarpetas")


PROJECT_ROOT = find_project_root()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
RESULTS_TABLES = PROJECT_ROOT / "results" / "tables"
RESULTS_FIGURES = PROJECT_ROOT / "results" / "figures"
RESULTS_MODELS = PROJECT_ROOT / "results" / "models"
for directory in (DATA_INTERIM, RESULTS_TABLES, RESULTS_FIGURES, RESULTS_MODELS):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
# Propuesta inicial: mitad temprana de validation para calibrar, mitad posterior
# para medir/seleccionar políticas. Registrar cualquier cambio antes de ejecutar.
VALIDATION_CALIBRATION_FRACTION = 0.50
MIN_CATEGORY_FREQUENCY = 50

# Baseline fijo y predeclarado; estos parámetros no se optimizan con validation.
BASE_MODEL_PARAMS = {
    "n_estimators": 250,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 10,
    "tree_method": "hist",
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "n_jobs": -1,
    "random_state": RANDOM_STATE,
    "verbosity": 0,
}

# Matriz acordada en la propuesta; falta fijar escenarios operativos de Ca.
COST_CONFIG = {
    "approved": False,
    "approval_reference": None,
    "amount_unit": None,
    "cost_unit": None,
    "administrative_cost_grid": None,
}

# Solo activar después de aprobar costos y congelar el protocolo/modelos.
RUN_FINAL_TEST = False
FINAL_TEST_PROTOCOL_FROZEN = False

if not 0 < VALIDATION_CALIBRATION_FRACTION < 1:
    raise ValueError("VALIDATION_CALIBRATION_FRACTION debe estar entre 0 y 1")
print(f"Proyecto: {PROJECT_ROOT}")
print(f"Python baseline: XGBoost; parámetros fijos: {BASE_MODEL_PARAMS}")
print(f"Validation: {VALIDATION_CALIBRATION_FRACTION:.0%} temprano para calibración")

Proyecto: C:\Users\Milton Cuervo\Documents\IEEE-CIS Fraud Detection-PI2
Python baseline: XGBoost; parámetros fijos: {'n_estimators': 250, 'max_depth': 6, 'learning_rate': 0.05, 'subsample': 0.8, 'colsample_bytree': 0.8, 'min_child_weight': 10, 'tree_method': 'hist', 'objective': 'binary:logistic', 'eval_metric': 'aucpr', 'n_jobs': -1, 'random_state': 42, 'verbosity': 0}
Validation: 50% temprano para calibración


## 2. Cargar datos y manifiesto temporal

Unimos las transacciones con la información de identidad y recuperamos la asignación de bloques por `TransactionID`. Verificamos cobertura, unicidad y orden temporal para que la población evaluada coincida con la partición documentada. Las etiquetas de test permanecen en el archivo original; evitamos usarlas para ajustar el modelo o sus transformaciones.


In [2]:
transactions, identity, merged = load_ieee_cis(DATA_RAW)
manifest_path = DATA_INTERIM / "split_manifest.parquet"
if not manifest_path.exists():
    raise FileNotFoundError(
        f"No existe {manifest_path}; ejecute primero notebooks/02_particion_temporal.ipynb"
    )
manifest = pd.read_parquet(manifest_path, columns=["TransactionID", "split"])
if manifest["TransactionID"].duplicated().any():
    raise ValueError("El manifiesto contiene TransactionID duplicados")

identity_ids = set(identity["TransactionID"])
merged["has_identity"] = merged["TransactionID"].isin(identity_ids).astype("int8")
frame = merged.merge(
    manifest,
    on="TransactionID",
    how="left",
    validate="one_to_one",
)
if len(frame) != len(transactions) or frame["split"].isna().any():
    raise AssertionError("El manifiesto no cubre exactamente las transacciones")
frame = frame.sort_values(["TransactionDT", "TransactionID"]).reset_index(drop=True)
validate_temporal_split(frame, frame["split"])

# No consultar ni imprimir etiquetas agregadas de test en el modelado.
development_rows = frame.loc[~frame["split"].eq("test")]
split_summary = development_rows.groupby("split", observed=True).agg(
    rows=("TransactionID", "size"),
    frauds=("isFraud", "sum"),
    fraud_rate=("isFraud", "mean"),
)
print(split_summary.reindex(["train", "gap_1", "validation", "gap_2"]))
test_window = frame.loc[frame["split"].eq("test"), "TransactionDT"]
print(f"Test reservado: {len(test_window):,} filas; ventana temporal {int(test_window.min())}–{int(test_window.max())}; sin resumen de etiquetas.")
assert set(frame["split"].unique()) == {"train", "gap_1", "validation", "gap_2", "test"}

              rows  frauds  fraud_rate
split                                 
train       380815   13011    0.034166
gap_1        20995    1015    0.048345
validation   84093    3075    0.036567
gap_2        20404     602    0.029504
Test reservado: 84,233 filas; ventana temporal 13270993–15811131; sin resumen de etiquetas.


## 3. Ventanas de train y validation; preparación sin fuga

Usamos train para aprender imputación y codificación. Dividimos validación cronológicamente: la ventana temprana ajusta el calibrador y la posterior, `validation_policy`, permite diagnosticar las probabilidades y explorar decisiones. Esta separación evita medir calibración sobre las mismas filas utilizadas para ajustarla. La validación posterior sigue siendo un bloque de desarrollo.

Al transformar nuevas operaciones debemos admitir categorías no observadas en train. Conservamos `TransactionAmt` en su escala original para evaluar costos, aunque el predictor utilice atributos derivados. Cada transformación aprendida debe conservar el mismo ajuste al aplicarse a validación y test.


In [3]:
# Los gaps se excluyen. Test no se extrae ni se puntúa en estas celdas.
train = frame.loc[frame["split"].eq("train")].copy()
validation = frame.loc[frame["split"].eq("validation")].copy()
validation = validation.sort_values(["TransactionDT", "TransactionID"]).reset_index(drop=True)
calibration_end = int(round(len(validation) * VALIDATION_CALIBRATION_FRACTION))
validation_cal = validation.iloc[:calibration_end].copy()
validation_policy = validation.iloc[calibration_end:].copy()

for name, block in (("train", train), ("validation_cal", validation_cal), ("validation_policy", validation_policy)):
    if block.empty or block["isFraud"].nunique() < 2:
        raise ValueError(f"{name} debe contener ambas clases para este protocolo")
    print(name, "rows=", len(block), "frauds=", int(block["isFraud"].sum()),
          "time_start=", int(block["TransactionDT"].min()),
          "time_end=", int(block["TransactionDT"].max()))

EXCLUDED_FEATURES = {"TransactionID", "TransactionDT", "isFraud", "split"}
CATEGORICAL_HINTS = {
    "ProductCD", "card1", "card2", "card3", "card4", "card5", "card6",
    "addr1", "addr2", "P_emaildomain", "R_emaildomain", "DeviceType", "DeviceInfo",
    *(f"M{i}" for i in range(1, 10)),
}


def make_features(data):
    X = data.drop(columns=[c for c in EXCLUDED_FEATURES if c in data.columns]).copy()
    for column in CATEGORICAL_HINTS.intersection(X.columns):
        X[column] = X[column].astype("string").fillna("__MISSING__")
    X = X.replace([np.inf, -np.inf], np.nan)
    return X


X_train = make_features(train)
X_cal = make_features(validation_cal)
X_policy = make_features(validation_policy)
y_train = train["isFraud"].astype("int8")
y_cal = validation_cal["isFraud"].astype("int8")
y_policy = validation_policy["isFraud"].astype("int8")

categorical_columns = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()
numeric_columns = [column for column in X_train.columns if column not in categorical_columns]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=MIN_CATEGORY_FREQUENCY,
        sparse_output=True,
        dtype=np.float32,
    )),
])
preprocessor = ColumnTransformer(
    [("numeric", numeric_pipeline, numeric_columns),
     ("categorical", categorical_pipeline, categorical_columns)],
    remainder="drop",
    sparse_threshold=0.3,
)

# Vocabulario, imputación y cualquier estadística se aprenden solo con train.
X_train_t = preprocessor.fit_transform(X_train)
X_cal_t = preprocessor.transform(X_cal)
X_policy_t = preprocessor.transform(X_policy)
print("Matriz train:", X_train_t.shape, "; variables numéricas:", len(numeric_columns),
      "; categóricas:", len(categorical_columns))

train rows= 380815 frauds= 13011 time_start= 86400 time_end= 9521218
validation_cal rows= 42046 frauds= 1558 time_start= 10126067 time_end= 11293017
validation_policy rows= 42047 frauds= 1517 time_start= 11293020 time_end= 12666174


Matriz train: (380815, 2621) ; variables numéricas: 395 ; categóricas: 37


## 4. Predictor base y calibración temporal

Entrenamos XGBoost con una configuración fija como primer modelo de referencia. Después ajustamos una regresión logística sobre el logit de sus probabilidades en la ventana temprana de validación; esta calibración sigmoidal busca mejorar la correspondencia entre probabilidades y frecuencias observadas.

Una puntuación alta ayuda a ordenar fraudes, pero la decisión económica utiliza el valor de la probabilidad. Por eso evaluamos discriminación y calibración. Mantenemos fijo el predictor que generó las puntuaciones del calibrador; reajustarlo después exigiría revisar el procedimiento de calibración.


In [4]:
base_model = XGBClassifier(**BASE_MODEL_PARAMS)
base_model.fit(X_train_t, y_train, verbose=False)


def clipped_logit(probabilities):
    p = np.clip(np.asarray(probabilities, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p)).reshape(-1, 1)


# Calibración Platt: el predictor no vio validation_cal ni validation_policy.
p_cal_raw = base_model.predict_proba(X_cal_t)[:, 1]
calibrator = LogisticRegression(C=1e6, solver="lbfgs", max_iter=1000)
calibrator.fit(clipped_logit(p_cal_raw), y_cal)
p_policy_raw = base_model.predict_proba(X_policy_t)[:, 1]
p_policy = calibrator.predict_proba(clipped_logit(p_policy_raw))[:, 1]

# Baseline fijo documentado; el umbral no se optimiza con validation.
action_fixed_policy = np.where(p_policy >= 0.5, "block", "approve")
print("Predictor entrenado con train; calibrador ajustado en validation temprana.")
print("No se han calculado acciones ni métricas de test.")

Predictor entrenado con train; calibrador ajustado en validation temprana.
No se han calculado acciones ni métricas de test.


## 5. Métricas predictivas de validation posterior

Comparamos las probabilidades originales y calibradas con average precision (AP), Brier score, log-loss y la curva de calibración. AP resume precision y recall; Brier y log-loss evalúan probabilidades. Ninguna métrica aislada demuestra que la calibración sea suficiente para todas las regiones de riesgo.

También calculamos precision y recall con el corte clasificatorio 0,5. Leemos estas salidas como diagnóstico del modelo de referencia, antes de interpretar ahorros. Las figuras y tablas que exportamos deben identificarse con la corrida que las produjo. AP no es necesariamente idéntica al área trapezoidal bajo la curva precision-recall.


In [5]:
validation_metrics = pd.DataFrame([{
    "block": "validation_policy",
    "rows": len(y_policy),
    "frauds": int(y_policy.sum()),
    "fraud_rate": float(y_policy.mean()),
    "average_precision": float(average_precision_score(y_policy, p_policy)),
    "recall_at_fixed_0_5": float(recall_score(y_policy, action_fixed_policy == "block")),
    "precision_at_fixed_0_5": float(precision_score(y_policy, action_fixed_policy == "block", zero_division=0)),
    "brier_score": float(brier_score_loss(y_policy, p_policy)),
    "log_loss": float(log_loss(y_policy, p_policy, labels=[0, 1])),
    "legitimate_block_rate_at_fixed_0_5": float(
        np.mean(action_fixed_policy[y_policy.to_numpy() == 0] == "block")
    ),
}])
RESULTS_TABLES.joinpath("modelo_metricas_validation.csv").parent.mkdir(parents=True, exist_ok=True)
validation_metrics.to_csv(RESULTS_TABLES / "modelo_metricas_validation.csv", index=False)

fraction_positive, mean_predicted = calibration_curve(y_policy, p_policy, n_bins=10, strategy="quantile")
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(mean_predicted, fraction_positive, marker="o", label="XGBoost + Platt")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Calibración ideal")
ax.set(xlabel="Probabilidad promedio", ylabel="Fracción observada de fraude",
       title="Calibración en validation posterior", xlim=(0, 1), ylim=(0, 1))
ax.legend()
fig.tight_layout()
fig.savefig(RESULTS_FIGURES / "modelo_calibracion_validation.png", dpi=160, bbox_inches="tight")
plt.show()

validation_scores = validation_policy[["TransactionID", "TransactionDT", "TransactionAmt", "isFraud"]].copy()
validation_scores["probability_raw"] = p_policy_raw
validation_scores["probability_calibrated"] = p_policy
validation_scores["action_fixed_0_5"] = action_fixed_policy
validation_scores.to_parquet(DATA_INTERIM / "validation_scores.parquet", index=False)
joblib.dump(
    {"preprocessor": preprocessor, "model": base_model, "calibrator": calibrator,
     "base_model_params": BASE_MODEL_PARAMS,
     "validation_calibration_fraction": VALIDATION_CALIBRATION_FRACTION},
    RESULTS_MODELS / "baseline_xgboost_platt.joblib",
)
display(validation_metrics)

C:\Users\Milton Cuervo\AppData\Local\Temp\ipykernel_23676\4029829512.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,block,rows,frauds,fraud_rate,average_precision,recall_at_fixed_0_5,precision_at_fixed_0_5,brier_score,log_loss,legitimate_block_rate_at_fixed_0_5
0,validation_policy,42047,1517,0.036079,0.468051,0.275544,0.768382,0.025004,0.101651,0.003109


## 6. Matriz de costos y referencias de ahorro

La matriz ya está definida en el anteproyecto. Lo pendiente es acordar valores de Ca expresados en la misma escala que el monto. Usamos funciones compartidas para que las tres estrategias se evalúen con idénticos costos.

El costo realizado tiene dos componentes: cantidad de intervenciones multiplicada por Ca, más el monto de fraudes aprobados. La referencia del ahorro es el menor costo entre aprobar todo e intervenir todo. También medimos la mejora frente a la política fija; son referencias distintas.


In [6]:
if COST_CONFIG["approved"]:
    CA_GRID = require_approved_costs(COST_CONFIG)
else:
    CA_GRID = ()
    print("Matriz definida; escenarios finales de Ca pendientes. Podemos explorar sensibilidad en notebook 04.")


Matriz definida; escenarios finales de Ca pendientes. Podemos explorar sensibilidad en notebook 04.


## 7. Comparación de las tres estrategias en validación

Por cada Ca acordado comparamos la regla fija, BMR y un árbol ponderado. El árbol aprende la acción más barata con la etiqueta conocida y pondera el error por su diferencia de costos: para fraude, |A_i - Ca|; para legítimas, Ca. Si A_i <= Ca, aprobar también es la acción de menor costo para un fraude conocido.

Esta conversión reproduce el costo empírico como error ponderado más una constante. El ajuste de un árbol mediante impureza sigue siendo una aproximación y no la implementación exacta de costcla. Cada Ca necesita su propio árbol porque cambia el problema de decisión.


In [7]:
cost_trees = {}
if CA_GRID:
    cost_tables = []
    for ca in CA_GRID:
        tree = fit_cost_tree(X_train_t, y_train, train["TransactionAmt"], ca, random_state=RANDOM_STATE)
        cost_trees[ca] = tree
        policies = {
            "fixed_0_5": action_fixed_policy,
            "bmr": bmr_actions(p_policy, validation_policy["TransactionAmt"], ca),
            "cost_sensitive_tree": np.where(tree.predict(X_policy_t) == 1, "block", "approve"),
        }
        table = evaluate_policies(y_policy, validation_policy["TransactionAmt"], policies, ca)
        table["evaluation_block"] = "validation_policy"
        cost_tables.append(table)
    validation_costs = pd.concat(cost_tables, ignore_index=True)
    validation_costs.to_csv(RESULTS_TABLES / "costos_validation_sensibilidad.csv", index=False)
    joblib.dump(cost_trees, RESULTS_MODELS / "cost_sensitive_trees.joblib")
    display(validation_costs)
else:
    print("Comparación final pendiente de escenarios acordados; use 04 para explorar Ca ilustrativo.")


Comparación final pendiente de escenarios acordados; use 04 para explorar Ca ilustrativo.


## 8. Evaluación temporal final

Activamos esta etapa una vez fijados predictor, calibrador, costos y reglas. Usamos las mismas filas para las tres estrategias. Los resultados no se reutilizan para modificar el modelo o elegir Ca.

Declararemos la consulta histórica de etiquetas agregadas del periodo. Esta reserva todavía evita seleccionar modelos con resultados predictivos de test, aunque no constituye un periodo estrictamente ciego desde el inicio.


In [8]:
if not RUN_FINAL_TEST:
    print("Test no puntuado: RUN_FINAL_TEST=False.")
else:
    final_ca_grid = require_approved_costs(COST_CONFIG)
    if not FINAL_TEST_PROTOCOL_FROZEN or set(cost_trees) != set(final_ca_grid):
        raise RuntimeError("Congele el protocolo y ajuste un árbol por Ca antes de evaluar test")
    test = frame.loc[frame["split"].eq("test")].copy()
    X_test_t = preprocessor.transform(make_features(test))
    p_test_raw = base_model.predict_proba(X_test_t)[:, 1]
    p_test = calibrator.predict_proba(clipped_logit(p_test_raw))[:, 1]
    test_tables = []
    for ca in final_ca_grid:
        policies = {
            "fixed_0_5": np.where(p_test >= 0.5, "block", "approve"),
            "bmr": bmr_actions(p_test, test["TransactionAmt"], ca),
            "cost_sensitive_tree": np.where(cost_trees[ca].predict(X_test_t) == 1, "block", "approve"),
        }
        table = evaluate_policies(test["isFraud"], test["TransactionAmt"], policies, ca)
        table["evaluation_block"] = "test"
        table["base_model_average_precision_common"] = average_precision_score(test["isFraud"], p_test)
        test_tables.append(table)
    test_costs = pd.concat(test_tables, ignore_index=True)
    test_costs.to_csv(RESULTS_TABLES / "costos_test_sensibilidad.csv", index=False)
    display(test_costs)


Test no puntuado: RUN_FINAL_TEST=False.


## 9. Lectura y límites

Interpretamos el costo como resultado de la matriz asumida, no como pérdida bancaria observada. La intervención evita la pérdida del monto en esta simplificación y cuesta Ca por operación. Su eficacia imperfecta queda fuera del experimento principal.

El ahorro puede ser negativo. BMR minimiza riesgo esperado con las probabilidades estimadas, pero no garantiza el menor costo realizado en cada muestra. Leemos costo, recall y bloqueo de legítimas conjuntamente. Las tres estrategias se comparan bajo la misma matriz; AP describe las probabilidades del modelo base y no cambia al variar su política.

La unidad monetaria aún necesita verificación. Los escenarios ilustrativos se expresan en unidades del dataset. La comparación principal varía Ca; no incorpora recuperación, costos B separados ni revisión como tercera acción.
